# Feature engineering causal

Inclui vizinhança horária, sazonalidades diária/semanal, janelas móveis deslocadas e clima defasado. RF e PLS usam as mesmas features.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
clean = pd.read_csv(
    require_file(config.DATA_DIR / "clean_hourly.csv", "02_limpeza_preparacao.ipynb"),
    parse_dates=[config.DATETIME_COL],
)
frame = features.modeling_matrix(features.add_calendar_and_lags(clean))
data_lib.save_frame(frame, config.DATA_DIR / "modeling_frame.csv")
protocol = models.protocol_summary(frame)
data_lib.save_json(protocol, config.RESULT_DIR / "protocol.json")
data_lib.save_json({
    "train_size_70pct": protocol["train_end_index"],
    "validation_end_80pct": protocol["validation_end_index"],
    "test_size_20pct": len(frame) - protocol["validation_end_index"],
    "features": config.FEATURE_COLS,
}, config.DATA_DIR / "split_info.json")
display(protocol)
display(frame.head())
print("features:", len(config.FEATURE_COLS))


{'protocol_version': 3,
 'random_state': 67,
 'environment': {'python': '3.11.9',
  'numpy': '2.1.3',
  'pandas': '2.2.3',
  'scikit_learn': '1.6.1',
  'statsmodels': '0.14.4'},
 'granularity': '1 hora',
 'horizon_hours': 1,
 'external_split': '80% desenvolvimento / 20% teste',
 'development_split': '70% treino interno / 10% validação',
 'train_end_index': 20044,
 'validation_end_index': 22908,
 'n_rows': 28636,
 'validation_origin_step_hours': 11,
 'test_origin_step_hours': 11,
 'n_validation_origins': 258,
 'n_test_origins': 521,
 'residual_diagnostic_hours': 168,
 'validation_origins': [20044,
  20055,
  20066,
  20077,
  20088,
  20099,
  20110,
  20121,
  20132,
  20143,
  20154,
  20165,
  20176,
  20187,
  20198,
  20209,
  20220,
  20231,
  20242,
  20253,
  20264,
  20275,
  20286,
  20297,
  20308,
  20319,
  20330,
  20341,
  20352,
  20363,
  20374,
  20385,
  20396,
  20407,
  20418,
  20429,
  20440,
  20451,
  20462,
  20473,
  20484,
  20495,
  20506,
  20517,
  20528,


,date_time,traffic_volume,target_observed,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,is_weekend,...,y_roll_mean_168,y_roll_std_168,temp_lag_1,rain_lag_1,snow_lag_1,clouds_lag_1,observed_lag_1,observed_lag_24,observed_lag_168,observed_lag_336
0,2015-06-25 20:00:00,3207.0,1,-0.866025,0.500000,0.433884,-0.900969,1.224647e-16,-1.0,0,...,2817.678571,1603.569926,298.52,0.0,0.0,1.0,1.0,1.0,0.0,1.0
1,2015-06-25 21:00:00,3189.0,1,-0.707107,0.707107,0.433884,-0.900969,1.224647e-16,-1.0,0,...,2819.589286,1603.843039,297.46,0.0,0.0,1.0,1.0,1.0,0.0,0.0
2,2015-06-25 22:00:00,2469.0,1,-0.500000,0.866025,0.433884,-0.900969,1.224647e-16,-1.0,0,...,2821.392857,1604.088515,295.25,0.0,0.0,1.0,1.0,1.0,0.0,0.0
3,2015-06-25 23:00:00,1507.0,1,-0.258819,0.965926,0.433884,-0.900969,1.224647e-16,-1.0,0,...,2818.910714,1604.310559,293.36,0.0,0.0,1.0,1.0,1.0,0.0,0.0
4,2015-06-26 00:00:00,865.0,1,0.000000,1.000000,-0.433884,-0.900969,1.224647e-16,-1.0,0,...,2810.702381,1607.489870,292.11,0.0,0.0,1.0,1.0,1.0,0.0,0.0


features: 36
